In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
from sklearn.impute import KNNImputer
import os 
import random
from scipy.interpolate import splrep, BSpline

In [ ]:
def decomposicao_svd(df):
    U, S, Vt = np.linalg.svd(df, full_matrices=True)
    return U, S, Vt

def grafico_variabilidade(variabilidade, S):
    plt.plot(range(1, len(variabilidade) + 1), variabilidade, marker='o', markersize=1, markerfacecolor='teal', markeredgecolor='teal', color='darkturquoise')
    plt.xlabel('Número de Valores Singulares')
    plt.ylabel('Variabilidade Acumulada')
    plt.title('Valores Singulares por Variabilidade Acumulada')
    plt.grid(color='lightgray', alpha=0.7)
    plt.show()

def componentes_principais(r, U, S, Vt):
    U_reduced = U[:, :r]
    S_reduced = S[:r]
    Vt_reduced = Vt[:r, :]
    return U_reduced, S_reduced, Vt_reduced
#trasnforma a matriz de volta em um dataframe
def matriztodf(dataframes):
    df = pd.DataFrame(dataframes)
    q = df.shape[0]*df.shape[1]
    df = df.transpose()
    df = df.to_numpy().reshape(-1, q)
    df = df.transpose()
    df = pd.DataFrame(df)
    if 0 in df.columns:
        df = df.rename(columns={0: 'Throughput'})
    return df

#calcula rmse dos valores gerados no svd com os valores 'originais'
def calcular_rmse(df1, df2, coluna):
    indices_comuns = df1.index.intersection(df2.index)
    valores_df1 = df1.loc[indices_comuns, coluna]
    valores_df2 = df2.loc[indices_comuns, coluna]
    rmse = np.sqrt(np.mean((valores_df1 - valores_df2) ** 2))
    return rmse

#gera arquivo csv final com a imputacao
def gerar_arq_csv(df1, df2, caminho_base, nome_arquivo_csv):
    df1['Throughput'] = pd.NA
    df1['Throughput'] = df2['Throughput']
    #exclui as ultimas linhas do arquivo que nao foi feita imputacao
    df1 = df1.dropna(subset=['Throughput'])
    caminho_svd = os.path.join(caminho_base, 'svd')
    if not os.path.exists(caminho_svd):
        os.makedirs(caminho_svd)
    caminho_arquivo_csv = os.path.join(caminho_svd, nome_arquivo_csv)
    # Salva o DataFrame resultante em um arquivo CSV
    df1.to_csv(caminho_arquivo_csv, index=False)
    print(f"Arquivo CSV '{caminho_arquivo_csv}' gerado com sucesso!")
    return df1

In [ ]:
#função para tratamento inicial do csv:
#corrigir aqui pq algo ta fazendo bugar o rj-ap

def decomposicao_svd(df):
    U, S, Vt = np.linalg.svd(df, full_matrices=True)
    return U, S, Vt

def grafico_variabilidade(variabilidade, S):
    plt.plot(range(1, len(variabilidade) + 1), variabilidade, marker='o', markersize=1, markerfacecolor='teal', markeredgecolor='teal', color='darkturquoise')
    plt.xlabel('Número de Valores Singulares')
    plt.ylabel('Variabilidade Acumulada')
    plt.title('Valores Singulares por Variabilidade Acumulada')
    plt.grid(color='lightgray', alpha=0.7)
    plt.show()

def componentes_principais(r, U, S, Vt):
    U_reduced = U[:, :r]
    S_reduced = S[:r]
    Vt_reduced = Vt[:r, :]
    return U_reduced, S_reduced, Vt_reduced
#trasnforma a matriz de volta em um dataframe
def matriztodf(dataframes):
    df = pd.DataFrame(dataframes)
    q = df.shape[0]*df.shape[1]
    df = df.transpose()
    df = df.to_numpy().reshape(-1, q)
    df = df.transpose()
    df = pd.DataFrame(df)
    if 0 in df.columns:
        df = df.rename(columns={0: 'Throughput'})
    return df

#calcula rmse dos valores gerados no svd com os valores 'originais'
def calcular_rmse(df1, df2, coluna):
    indices_comuns = df1.index.intersection(df2.index)
    valores_df1 = df1.loc[indices_comuns, coluna]
    valores_df2 = df2.loc[indices_comuns, coluna]
    rmse = np.sqrt(np.mean((valores_df1 - valores_df2) ** 2))
    return rmse

#gera arquivo csv final com a imputacao
def gerar_arq_csv(df1, df2, caminho_base, nome_arquivo_csv):
    df1['Throughput'] = pd.NA
    df1['Throughput'] = df2['Throughput']
    #exclui as ultimas linhas do arquivo que nao foi feita imputacao
    df1 = df1.dropna(subset=['Throughput'])
    caminho_svd = os.path.join(caminho_base, 'svd')
    if not os.path.exists(caminho_svd):
        os.makedirs(caminho_svd)
    caminho_arquivo_csv = os.path.join(caminho_svd, nome_arquivo_csv)
    # Salva o DataFrame resultante em um arquivo CSV
    df1.to_csv(caminho_arquivo_csv, index=False)
    print(f"Arquivo CSV '{caminho_arquivo_csv}' gerado com sucesso!")
    return df1

def outlier_removal(df, column):

    df[column] = df[column].replace(-1, np.nan)

    r = df[column].dropna().to_numpy()
    
    if r.size == 0:
        print("Coluna não contém valores suficientes para análise.")
        return df

    r_max = np.max(r) 
    r = r / r_max  

    perc_min = []
    p_min = np.linspace(0.1, 2, 20)
    for i in p_min:
        perc_min.append(np.percentile(r, i))
    diff_perc_min = np.diff(perc_min)
    index_min = np.argmax(diff_perc_min)  
    thres_min = np.mean(perc_min[index_min:index_min + 2])

    perc_max = []
    p_max = np.linspace(98, 100, 20)
    for i in p_max:
        perc_max.append(np.percentile(r, i))
    diff_perc_max = np.diff(perc_max)
    index_max = np.argmax(diff_perc_max)  
    thres_max = np.mean(perc_max[index_max:index_max + 2])

    r_filtered = np.where((r < thres_min) | (r > thres_max), np.NaN, r)

    r_filtered = r_filtered * r_max  

    df_filtered = df.copy()
    df_filtered.loc[~df[column].isna(), column] = r_filtered

    return df_filtered

def matriz(path):
    df = pd.read_csv(path)
    df = outlier_removal(df, 'Throughput')
    df_datetime = df.copy()
    df_datetime.drop(columns=['Throughput'], inplace = True)
    df['Throughput'] = df['Throughput'].replace(-1, np.nan)
    
    Throughput = df['Throughput'].values
    num_dados = len(Throughput)
    num_colunas = num_dados // 28
    matriz = Throughput[:num_colunas*28].reshape(num_colunas, 28).T
    matriz_original = pd.DataFrame(matriz)
    df_interpolado = df["Throughput"].interpolate(method='linear', limit_direction='both')
    Throughput_=df_interpolado.values
    matriz_interpolado = Throughput_[:num_colunas*28].reshape(num_colunas, 28).T
    matriz_interpolado= pd.DataFrame(matriz_interpolado)
    mask = np.isnan(matriz_original.values)
    matriz_mascara = pd.DataFrame(mask)
    return matriz_original, matriz_mascara, matriz_interpolado, df_datetime#, r_max

In [ ]:
caminhos_csv = ['datasets/throughput/treated/cubic/treated cubic esmond data ap-ba 07-03-2023.csv', 'datasets/throughput/treated/cubic/treated cubic esmond data ba-ap 07-08-2023.csv', 'datasets/throughput/treated/cubic/treated cubic esmond data ba-rj 07-03-2023.csv','datasets/throughput/treated/cubic/treated cubic esmond data es-ap 07-08-2023.csv', 'datasets/throughput/treated/cubic/treated cubic esmond data es-rs 07-03-2023.csv']

In [ ]:
caminhos_csv = [r'datasets\throughput\longest interval\treated bbr esmond data ac-pa 07-03-2023_longest_interval.csv', r'datasets\throughput\longest interval\treated bbr esmond data ap-ba 07-03-2023_longest_interval.csv', r'datasets\throughput\longest interval\treated bbr esmond data ap-rs 07-03-2023_longest_interval.csv', r'datasets\throughput\longest interval\treated bbr esmond data go-es 07-08-2023_longest_interval.csv', r'datasets\throughput\longest interval\treated bbr esmond data rs-ce 07-07-2023_longest_interval.csv', r'datasets\throughput\longest interval\treated bbr esmond data rs-go 07-07-2023_longest_interval.csv', r'datasets\throughput\longest interval\treated cubic esmond data ap-ce 07-08-2023_longest_interval.csv', r'datasets\throughput\longest interval\treated cubic esmond data ap-rn 07-03-2023_longest_interval.csv', r'datasets\throughput\longest interval\treated cubic esmond data ap-rs 07-03-2023_longest_interval.csv', r'datasets\throughput\longest interval\treated cubic esmond data rs-es 07-07-2023_longest_interval.csv']
resultados = {}

for caminho_csv in caminhos_csv:
    nome_arquivo = os.path.basename(caminho_csv)
    
    resultados[nome_arquivo] = {'interpolacao_linear': None, 'svd_final': None}
    df_matriz, df_mask, df_interpolado, df_datetime = matriz(caminho_csv)
    resultados[nome_arquivo]['interpolacao_linear'] = df_interpolado.copy()

    A_anterior = df_interpolado.values.copy()
    rmse = float('inf') 
    max_iter = 300
    n_iter = 0


    while rmse >= 1e-3 and n_iter<=max_iter:  
        U, S, Vt = decomposicao_svd(df_interpolado)
        variabilidade = np.cumsum(S**2) / np.sum(S**2)

        porcentagem_variabilidade = 0.95
        r = np.where(variabilidade >= porcentagem_variabilidade)[0][0] + 1
        
        #print(f'Número de valores singulares para atingir {porcentagem_variabilidade*100}% de variabilidade: {r}')

        U_reduzido, S_reduzido, Vt_reduzido = componentes_principais(r, U, S, Vt)
        S_reduzido_matriz = np.diag(S_reduzido)

        A_aproximada = np.dot(np.dot(U_reduzido, S_reduzido_matriz), Vt_reduzido)
        A_aproximada_df = pd.DataFrame(A_aproximada)

        df_matriz_preenchida = df_matriz.fillna(A_aproximada_df)
        
        resultados[nome_arquivo]['svd_final'] = df_matriz_preenchida

        # Atualiza df_interpolado para a próxima iteração
        df_interpolado = df_matriz_preenchida.values
        
        # Calcular o RMSE entre a matriz atual e a anterior
        rmse = np.sqrt(np.mean((A_aproximada - A_anterior) ** 2))

        # Atualiza A_anterior para a próxima comparação
        A_anterior = A_aproximada.copy()

        n_iter +=1

    print(f'RMSE na iteração atual: {rmse}')
    print(f'Finalizando processamento para {caminho_csv}')

    svd = matriztodf(resultados[nome_arquivo]["svd_final"])
    interpolacao = matriztodf(resultados[nome_arquivo]["interpolacao_linear"])
    mask = matriztodf(df_mask)
    dfs_reshaped = []
    dfs_reshaped.append(interpolacao) #0
    dfs_reshaped.append(svd) #1
    dfs_reshaped.append(mask) #2
    # Chama a função para plotar os dados
    # plot_imputed_data(dfs_reshaped)
    base_path = 'datasets/imputed-longest-interval'
    gerar_arq_csv(df_datetime, dfs_reshaped[1], base_path, nome_arquivo)


In [ ]:
def impute_knn(df, k=5):
    imputer = KNNImputer(n_neighbors=k)
    df['Throughput'] = imputer.fit_transform(df[['Throughput']])
    return df

def impute_rolling_median(df, window_size=3):
    df['Throughput'] = df['Throughput'].fillna(df['Throughput'].rolling(window=window_size, min_periods=1).median())
    return df

def impute_rolling_average(df, window_size=3):
    df['Throughput'] = df['Throughput'].fillna(df['Throughput'].rolling(window=window_size, min_periods=1).mean())
    return df

In [ ]:
def data_imputation_datasets(datasets_path_list, imputation_column='Throughput'):
    list_imputed_dfs = []
    for path_dataset in datasets_path_list: 
        original_df = pd.read_csv(path_dataset)
        df = original_df.copy()

        # Removing outliers for datasets that are being imputed with knn, linear interp and moving average and median
        df = outlier_removal(df, imputation_column) 

        df_imputed_linear_interpolation = df.copy().interpolate(method='linear', limit_direction='both')

        list_imputed_dfs.append(df_imputed_linear_interpolation)

        df_imputed_knn = impute_knn(df.copy(), imputation_column=imputation_column)

        list_imputed_dfs.append(df_imputed_knn)

        df_imputed_rolling_average = impute_rolling_average(df.copy(), imputation_column=imputation_column)

        list_imputed_dfs.append(df_imputed_rolling_average)

        df_imputed_rolling_median = impute_rolling_median(df.copy(), imputation_column=imputation_column)

        list_imputed_dfs.append(df_imputed_rolling_median)

        # For specified pre processing for svd, the path is given, not the already without outlier df
        df_imputed_svd = impute_svd(path_dataset)

        list_imputed_dfs.append(df_imputed_svd)

        return list_imputed_dfs

def impute_knn(df, imputation_column, k=5):
    imputer = KNNImputer(n_neighbors=k)
    df[imputation_column] = imputer.fit_transform(df[[imputation_column]])
    return df

def impute_rolling_median(df, imputation_column, window_size=3):
    df[imputation_column] = df[imputation_column].fillna(df[imputation_column].rolling(window=window_size, min_periods=1).median())
    return df

def impute_rolling_average(df, imputation_column, window_size=3):
    df[imputation_column] = df[imputation_column].fillna(df[imputation_column].rolling(window=window_size, min_periods=1).mean())
    return df 

def outlier_removal(df, column):

    df[column] = df[column].replace(-1, np.nan)

    r = df[column].dropna().to_numpy()
    
    if r.size == 0:
        print("Column does not contain enough values ​​for analysis.")
        return df

    r_max = np.max(r) 
    r = r / r_max  

    perc_min = []
    p_min = np.linspace(0.1, 2, 20)
    for i in p_min:
        perc_min.append(np.percentile(r, i))
    diff_perc_min = np.diff(perc_min)
    index_min = np.argmax(diff_perc_min)  
    thres_min = np.mean(perc_min[index_min:index_min + 2])

    perc_max = []
    p_max = np.linspace(98, 100, 20)
    for i in p_max:
        perc_max.append(np.percentile(r, i))
    diff_perc_max = np.diff(perc_max)
    index_max = np.argmax(diff_perc_max)  
    thres_max = np.mean(perc_max[index_max:index_max + 2])

    r_filtered = np.where((r < thres_min) | (r > thres_max), np.NaN, r)

    r_filtered = r_filtered * r_max  

    df_filtered = df.copy()
    df_filtered.loc[~df[column].isna(), column] = r_filtered

    return df_filtered


def impute_svd(path_dataset):
    def decomposicao_svd(df):
        U, S, Vt = np.linalg.svd(df, full_matrices=True)
        return U, S, Vt

    # def grafico_variabilidade(variabilidade, S):
    #     plt.plot(range(1, len(variabilidade) + 1), variabilidade, marker='o', markersize=1, markerfacecolor='teal', markeredgecolor='teal', color='darkturquoise')
    #     plt.xlabel('Número de Valores Singulares')
    #     plt.ylabel('Variabilidade Acumulada')
    #     plt.title('Valores Singulares por Variabilidade Acumulada')
    #     plt.grid(color='lightgray', alpha=0.7)
    #     plt.show()

    def componentes_principais(r, U, S, Vt):
        U_reduced = U[:, :r]
        S_reduced = S[:r]
        Vt_reduced = Vt[:r, :]
        return U_reduced, S_reduced, Vt_reduced
    #trasnforma a matriz de volta em um dataframe
    def matriztodf(dataframes):
        df = pd.DataFrame(dataframes)
        q = df.shape[0]*df.shape[1]
        df = df.transpose()
        df = df.to_numpy().reshape(-1, q)
        df = df.transpose()
        df = pd.DataFrame(df)
        if 0 in df.columns:
            df = df.rename(columns={0: 'Throughput'})
        return df

    #calcula rmse dos valores gerados no svd com os valores 'originais'
    # def calcular_rmse(df1, df2, coluna):
    #     indices_comuns = df1.index.intersection(df2.index)
    #     valores_df1 = df1.loc[indices_comuns, coluna]
    #     valores_df2 = df2.loc[indices_comuns, coluna]
    #     rmse = np.sqrt(np.mean((valores_df1 - valores_df2) ** 2))
    #     return rmse

    #gera arquivo csv final com a imputacao
    def gerar_arq_csv(df1, df2, caminho_base, nome_arquivo_csv):
        df1['Throughput'] = pd.NA
        df1['Throughput'] = df2['Throughput']
        #exclui as ultimas linhas do arquivo que nao foi feita imputacao
        df1 = df1.dropna(subset=['Throughput'])
        caminho_svd = os.path.join(caminho_base, 'svd')
        if not os.path.exists(caminho_svd):
            os.makedirs(caminho_svd)
        caminho_arquivo_csv = os.path.join(caminho_svd, nome_arquivo_csv)
        # Salva o DataFrame resultante em um arquivo CSV
        df1.to_csv(caminho_arquivo_csv, index=False)
        print(f"Arquivo CSV '{caminho_arquivo_csv}' gerado com sucesso!")
        return df1

    def matriz(path):
        df = pd.read_csv(path)
        df = outlier_removal(df, 'Throughput')
        df_datetime = df.copy()
        df_datetime.drop(columns=['Throughput'], inplace = True)
        df['Throughput'] = df['Throughput'].replace(-1, np.nan)
        
        Throughput = df['Throughput'].values
        num_dados = len(Throughput)
        num_colunas = num_dados // 28
        matriz = Throughput[:num_colunas*28].reshape(num_colunas, 28).T
        matriz_original = pd.DataFrame(matriz)
        df_interpolado = df["Throughput"].interpolate(method='linear', limit_direction='both')
        Throughput_=df_interpolado.values
        matriz_interpolado = Throughput_[:num_colunas*28].reshape(num_colunas, 28).T
        matriz_interpolado= pd.DataFrame(matriz_interpolado)
        mask = np.isnan(matriz_original.values)
        matriz_mascara = pd.DataFrame(mask)
        return matriz_original, matriz_mascara, matriz_interpolado, df_datetime#, r_max
    
    nome_arquivo = os.path.basename(path_dataset)
    
    resultados[nome_arquivo] = {'interpolacao_linear': None, 'svd_final': None}
    df_matriz, df_mask, df_interpolado, df_datetime = matriz(path_dataset)
    resultados[nome_arquivo]['interpolacao_linear'] = df_interpolado.copy()

    A_anterior = df_interpolado.values.copy()
    rmse = float('inf') 
    max_iter = 300
    n_iter = 0

    while rmse >= 1e-3 and n_iter<=max_iter:  
        U, S, Vt = decomposicao_svd(df_interpolado)
        variabilidade = np.cumsum(S**2) / np.sum(S**2)

        porcentagem_variabilidade = 0.95
        r = np.where(variabilidade >= porcentagem_variabilidade)[0][0] + 1
        
        #print(f'Número de valores singulares para atingir {porcentagem_variabilidade*100}% de variabilidade: {r}')

        U_reduzido, S_reduzido, Vt_reduzido = componentes_principais(r, U, S, Vt)
        S_reduzido_matriz = np.diag(S_reduzido)

        A_aproximada = np.dot(np.dot(U_reduzido, S_reduzido_matriz), Vt_reduzido)
        A_aproximada_df = pd.DataFrame(A_aproximada)

        df_matriz_preenchida = df_matriz.fillna(A_aproximada_df)
        
        resultados[nome_arquivo]['svd_final'] = df_matriz_preenchida

        # Atualiza df_interpolado para a próxima iteração
        df_interpolado = df_matriz_preenchida.values
        
        # Calcular o RMSE entre a matriz atual e a anterior
        rmse = np.sqrt(np.mean((A_aproximada - A_anterior) ** 2))

        # Atualiza A_anterior para a próxima comparação
        A_anterior = A_aproximada.copy()

        n_iter +=1

    # print(f'RMSE na iteração atual: {rmse}')
    # print(f'Finalizando processamento para {caminho_csv}')

    svd = matriztodf(resultados[nome_arquivo]["svd_final"])
    # interpolacao = matriztodf(resultados[nome_arquivo]["interpolacao_linear"])
    # mask = matriztodf(df_mask)
    # dfs_reshaped = []
    # dfs_reshaped.append(interpolacao) #0
    # dfs_reshaped.append(svd) #1
    # dfs_reshaped.append(mask) #2
    # Chama a função para plotar os dados
    # plot_imputed_data(dfs_reshaped)
    # base_path = 'datasets/dados-vazao-imputados/svd'
    # gerar_arq_csv(df_datetime, dfs_reshaped[1], base_path, nome_arquivo)
    return svd

In [ ]:
datasets_path = ['datasets/throughput/treated/bbr/treated bbr esmond data ap-ba 07-03-2023.csv', 'datasets/throughput/treated/bbr/treated bbr esmond data ba-ap 07-08-2023.csv', 'datasets/throughput/treated/bbr/treated bbr esmond data ba-rj 07-03-2023.csv','datasets/throughput/treated/bbr/treated bbr esmond data es-ap 07-08-2023.csv', 'datasets/throughput/treated/bbr/treated bbr esmond data es-rs 07-03-2023.csv']

data_imputation_datasets(datasets_path)